# 07 — 调试与最佳实践

**来源:** [菜鸟教程 — LangGraph 入门教程](https://www.runoob.com/ai-agent/langgraph-quick-start.html)

LangGraph 的调试工具和工程最佳实践。

## 1. LangGraph Studio 可视化调试

LangGraph Studio 是官方提供的可视化开发环境，可实时查看 Agent 执行过程。

### 安装
```bash
pip install langgraph-cli -i https://mirrors.aliyun.com/pypi/simple/
```

### 创建配置文件 `langgraph.json`
```json
{
    "dependencies": ["."],
    "graphs": {
        "my_agent": "./my_agent.py:graph"
    },
    "env": ".env"
}
```

### 启动
```bash
langgraph dev
```

启动后访问 http://localhost:8123 即可在浏览器中使用。

**主要功能：**
- 实时可视化：图形化展示节点执行过程
- 状态检查：在任意节点暂停查看当前 State
- 时间旅行：回放历史执行步骤
- 热重载：修改代码后自动更新图结构

> **环境要求：** 需要 Docker 环境支持。

## 2. 最佳实践

### 状态设计要点
- 保持 State 精简，只包含必要字段
- 为复杂字段定义明确的 reducer（如 `add_messages`）
- 使用 Pydantic 模型在生产环境中验证状态类型
- 避免在 State 中存储过大的对象，考虑使用外部存储

### 节点设计要点
- 每个节点职责单一，便于测试和复用
- 节点函数应该是幂等的（相同输入产生相同输出）
- 避免直接修改传入的 state，而是返回新值
- 合理使用异步节点处理 I/O 密集型操作

### 错误处理

In [ ]:
def robust_node(state: dict) -> dict:
    """带错误处理的节点"""
    try:
        result = risky_operation(state)
        return {"messages": [result], "error": None}
    except Exception as e:
        return {
            "error": str(e),
            "messages": [{"role": "assistant", "content": f"操作失败: {e}"}]
        }


def risky_operation(state):
    # 模拟可能失败的操作
    return "操作成功"


print("带错误处理的节点定义完成")


### 避免无限循环

In [ ]:
def route_with_limit(state: dict) -> str:
    """带重试限制的路由函数，防止无限循环"""
    # 设置最大重试次数
    if state.get("retry_count", 0) >= 3:
        return "__end__"
    if needs_retry(state):
        return "retry_node"
    return "__end__"


def needs_retry(state):
    return False


print("带限制的路由函数定义完成")


## 3. 常见问题 FAQ

**Q1：节点返回值格式不对怎么办？**

In [ ]:
# 错误：直接修改 state 对象
def bad_node(state):
    state["messages"].append(...)  # 不要直接修改
    return state

# 正确：返回需要更新的字段
def good_node(state):
    new_message = ""
    return {"messages": [new_message]}  # 只返回变更字段

**Q2：如何在节点之间传递临时数据？**

将临时数据加入 State 定义，或使用下划线前缀约定为内部字段：

In [ ]:
from typing import TypedDict

class PublicState(TypedDict):
    messages: list  # 对外暴露

class PrivateState(TypedDict):
    messages: list
    _internal_cache: dict  # 以下划线开头约定为内部使用

**Q3：StateGraph 和 MessageGraph 的区别？**
```
# 使用 stream 模式观察每个节点的输出
for event in graph.stream(initial_state, stream_mode="updates"):
    for node_name, state_update in event.items():
        print(f"\n[节点: {node_name}]")
        print(f"更新: {state_update}")
```

**Q4：StateGraph 和 MessageGraph 的区别？**

MessageGraph 是早期版本的 API，功能较为受限。现在推荐统一使用 StateGraph，它更灵活、功能更完善。如需处理消息，使用 StateGraph(MessagesState) 或自定义包含 messages 字段的 State。


MessageGraph 已被废弃，请统一使用 `StateGraph(MessagesState)`。

---

**参考链接**

- [LangGraph GitHub 仓库](https://github.com/langchain-ai/langgraph)
- [菜鸟教程 — LangGraph 入门教程](https://www.runoob.com/ai-agent/langgraph-quick-start.html)
- [LangChain 官方文档](https://langchain-ai.github.io/langgraph/)
